# Case 1 — Dynamic data retrieval

Validação da função `retrieve_data(engine, product_code, store_code, date_range)` (ver [src/retrieve_data.py](../src/retrieve_data.py)) contra o banco real, cobrindo as combinações de filtro que outros times poderiam usar, os casos de erro, e o contexto de escala da tabela `data_product_sales`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from sqlalchemy import text
from IPython.display import display

from src.database import create_database_engine
from src.retrieve_data import retrieve_data

In [2]:
engine = create_database_engine()

# Exemplo A: apenas date_range
data = retrieve_data(
    engine,
    product_code=None,
    store_code=None,
    date_range=['2019-01-01', '2019-01-31'],
)
print(f"Exemplo A (apenas date_range): {len(data):,} linhas")
data.head()

Exemplo A (apenas date_range): 39,401 linhas


,STORE_CODE,PRODUCT_CODE,DATE,SALES_VALUE,SALES_QTY
0,1,18,2019-01-01,708.5,65.0
1,1,18,2019-01-02,1297.1,119.0
2,1,18,2019-01-03,1144.5,105.0
3,1,18,2019-01-04,1090.0,100.0
4,1,18,2019-01-05,893.8,82.0


## Combinações de filtro testadas

- **B — apenas `product_code`**: confirma que a função filtra por produto isoladamente.
- **C — apenas `store_code`**: confirma o filtro por loja isolado. Vale notar que `STORE_CODE` está tipado como `VARCHAR(255)` em `data_product_sales` (diferente de `data_store_cad`/`data_store_sales`, onde é `INTEGER`) — testamos de propósito passando um `int` para garantir que a comparação parametrizada funciona mesmo com essa inconsistência de schema.
- **D — os três filtros juntos**: cruza com a contagem esperada (31 dias de janeiro/2019) para validar que o `AND` entre cláusulas está correto.
- **E — entradas inválidas**: confirma que a função falha de forma clara (`ValueError`) em vez de silenciosamente devolver dados errados, para os três jeitos mais prováveis de um outro time errar a chamada.
- **Contexto de escala**: sem nenhum filtro, a função devolveria a tabela inteira — por isso não executamos essa chamada aqui (evitar puxar ~2,17M linhas só para demonstração).

In [3]:
print("=== Exemplo B: apenas product_code ===")
data_product_only = retrieve_data(engine, product_code=18, store_code=None, date_range=None)
print(f"{len(data_product_only):,} linhas")
display(data_product_only.head())

print("\n=== Exemplo C: apenas store_code ===")
data_store_only = retrieve_data(engine, product_code=None, store_code=1, date_range=None)
print(f"{len(data_store_only):,} linhas (STORE_CODE é VARCHAR em data_product_sales; o MySQL converte o parâmetro inteiro na comparação)")
display(data_store_only.head())

print("\n=== Exemplo D: product_code + store_code + date_range combinados ===")
data_combined = retrieve_data(engine, product_code=18, store_code=1, date_range=['2019-01-01', '2019-01-31'])
print(f"{len(data_combined)} linhas (esperado: 31, uma por dia de janeiro/2019)")
display(data_combined)

print("\n=== Exemplo E: validação de entradas inválidas ===")
casos_invalidos = [
    (dict(date_range=['2019-01-31']), "lista de data com 1 elemento"),
    (dict(date_range=['2019-01-31', '2019-01-01']), "data inicial depois da final"),
    (dict(date_range=['31-01-2019', '2019-01-31']), "formato de data fora do ISO"),
]
for kwargs, motivo in casos_invalidos:
    try:
        retrieve_data(engine, **kwargs)
        print(f"[FALHOU] deveria ter levantado erro para: {motivo}")
    except ValueError as exc:
        print(f"[OK] erro esperado para {motivo}: {exc}")

print("\n=== Contexto de escala: chamada sem nenhum filtro ===")
with engine.connect() as connection:
    total_rows = connection.execute(text("SELECT COUNT(*) FROM data_product_sales")).scalar()
    min_date, max_date = connection.execute(text("SELECT MIN(DATE), MAX(DATE) FROM data_product_sales")).one()
print(f"Sem filtros, retrieve_data devolveria as {total_rows:,} linhas da tabela inteira, cobrindo {min_date} a {max_date}.")

=== Exemplo B: apenas product_code ===


5,840 linhas


,STORE_CODE,PRODUCT_CODE,DATE,SALES_VALUE,SALES_QTY
0,1,18,2019-01-01,708.5,65.0
1,1,18,2019-01-02,1297.1,119.0
2,1,18,2019-01-03,1144.5,105.0
3,1,18,2019-01-04,1090.0,100.0
4,1,18,2019-01-05,893.8,82.0



=== Exemplo C: apenas store_code ===


136,729 linhas (STORE_CODE é VARCHAR em data_product_sales; o MySQL converte o parâmetro inteiro na comparação)


...


=== Exemplo D: product_code + store_code + date_range combinados ===


31 linhas (esperado: 31, uma por dia de janeiro/2019)


,STORE_CODE,PRODUCT_CODE,DATE,SALES_VALUE,SALES_QTY
0,1,18,2019-01-01,708.5,65.0
1,1,18,2019-01-02,1297.1,119.0
2,1,18,2019-01-03,1144.5,105.0
3,1,18,2019-01-04,1090.0,100.0
4,1,18,2019-01-05,893.8,82.0
5,1,18,2019-01-06,741.2,68.0
6,1,18,2019-01-07,654.0,60.0
7,1,18,2019-01-08,741.2,68.0
8,1,18,2019-01-09,1373.4,126.0
9,1,18,2019-01-10,1068.2,98.0



=== Exemplo E: validação de entradas inválidas ===
[OK] erro esperado para lista de data com 1 elemento: date_range must contain exactly a start date and an end date
[OK] erro esperado para data inicial depois da final: date_range start date cannot be after the end date
[OK] erro esperado para formato de data fora do ISO: Invalid isoformat string: '31-01-2019'

=== Contexto de escala: chamada sem nenhum filtro ===


Sem filtros, retrieve_data devolveria as 2,173,133 linhas da tabela inteira, cobrindo 2014-11-23 a 2019-12-31.


## Validação final

Testamos as combinações relevantes de parâmetros (nenhum time vai sempre passar os três filtros): só `product_code`, só `store_code`, os três juntos, e entradas malformadas.

- Em todos os casos a contagem de linhas retornada bate com o esperado — no Exemplo D em particular, a contagem exata (31 linhas para 31 dias de janeiro) confirma que os filtros são combinados com `AND` e não substituem uns aos outros.
- A parametrização via `:product_code`/`:store_code`/`:start_date`/`:end_date` (bind parameters do SQLAlchemy) evita concatenar valores diretamente na query — importante porque a função foi desenhada para ser chamada por outros times, então os valores de entrada não são confiáveis por padrão.
- A validação de `date_range` (tamanho da lista, ordem cronológica, formato ISO) rejeita entradas malformadas com uma mensagem clara em vez de gerar uma query quebrada ou um resultado vazio silencioso.
- `SELECT *` garante que todas as colunas de `data_product_sales` são retornadas, conforme pedido no enunciado.
- `engine` é recebido como parâmetro (não criado dentro da função), permitindo que quem chama reutilize a mesma conexão entre múltiplas chamadas.